# OronTTS — Mongolian & Kazakh TTS (F5-TTS)

Train and run inference on **Google Colab Free** (T4 GPU, 15 GB VRAM).

**What this does:**
1. Installs OronTTS from GitHub
2. Loads `btsee/mbspeech_mn` dataset from HuggingFace
3. Trains F5-TTS Small on T4
4. Runs inference and plays audio

> **Tip:** Go to `Runtime → Change runtime type → T4 GPU` before running.

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Clone and install
!git clone https://github.com/btsee/oron-tts.git /content/oron-tts
%cd /content/oron-tts
!pip install -e . -q

In [ ]:
# (Optional) Mount Google Drive for persistent checkpoints
MOUNT_DRIVE = True  # @param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/oron-tts/checkpoints"
    OUTPUT_DIR = "/content/drive/MyDrive/oron-tts/outputs"
else:
    CHECKPOINT_DIR = "/content/oron-tts/output/checkpoints"
    OUTPUT_DIR = "/content/oron-tts/output"

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Outputs:     {OUTPUT_DIR}")

In [ ]:
# (Optional) HuggingFace token for private datasets or pushing models
HF_TOKEN = ""  # @param {type:"string"}

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    !huggingface-cli login --token $HF_TOKEN

## 2. Verify Installation

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Quick pipeline smoke test
from src.utils.tokenizer import CyrillicTokenizer
from src.utils.text_cleaner import TextCleaner

tok = CyrillicTokenizer()
tc = TextCleaner()

test_text = "Сайн байна уу, 2026 он"
cleaned = tc.clean(test_text, lang="mn")
ids = tok.encode(cleaned, lang="mn")
print(f"Input:   {test_text}")
print(f"Cleaned: {cleaned}")
print(f"IDs:     {ids[:10]}... (len={len(ids)})")
print(f"Vocab:   {tok.vocab_size}")

## 3. Training

Uses F5-TTS Small (dim=512, depth=12, ~400K params) — fits easily on T4.

Dataset: `btsee/mbspeech_mn` (3,846 samples, loaded from HuggingFace).

In [ ]:
# Write Colab-optimized config
COLAB_CONFIG = "/content/oron-tts/configs/colab.yaml"

config_text = """\
# OronTTS F5-TTS Colab Free Configuration
# Hardware: T4 GPU (15 GB VRAM), ~12 GB RAM

# == Audio ==
sample_rate: 24000
n_fft: 1024
hop_length: 256
win_length: 1024
n_mels: 100
fmin: 0.0
fmax: 8000.0

# == Training (conservative for Colab Free) ==
batch_size: 4
num_epochs: 100
learning_rate: 1.0e-4
betas: [0.9, 0.999]
warmup_steps: 500
ema_decay: 0.9999
max_grad_norm: 1.0
grad_accumulation_steps: 4
num_workers: 2
pin_memory: true
log_interval: 50
save_interval: 5
use_tqdm: true

# CUDA (T4)
cudnn_benchmark: true
use_tf32: false
compile: false

# == Model (F5-TTS Small) ==
model:
  vocab_size: 65
  dim: 512
  depth: 12
  heads: 8
  ff_mult: 4
  text_dim: 256
  conv_layers: 4
  p_dropout: 0.1
  vocos_dim: 384
  vocos_layers: 6
  vocos_intermediate: 1024
"""

with open(COLAB_CONFIG, "w") as f:
    f.write(config_text)
print(f"Config written to {COLAB_CONFIG}")

In [ ]:
# Train
!python scripts/train.py \
    --config configs/colab.yaml \
    --dataset btsee/mbspeech_mn \
    --lang mn \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --log-dir /content/oron-tts/output/logs \
    --num-epochs 100

In [ ]:
# Resume training (if Colab disconnected)
# !python scripts/train.py \
#     --config configs/colab.yaml \
#     --dataset btsee/mbspeech_mn \
#     --lang mn \
#     --checkpoint-dir {CHECKPOINT_DIR} \
#     --log-dir /content/oron-tts/output/logs \
#     --resume

## 4. Inference

In [ ]:
import glob

# Find latest checkpoint
ckpts = sorted(glob.glob(f"{CHECKPOINT_DIR}/f5tts_step_*.pt"))
if not ckpts:
    print("No checkpoints found. Train first.")
else:
    LATEST_CKPT = ckpts[-1]
    print(f"Using checkpoint: {LATEST_CKPT}")

In [ ]:
# Synthesize
TEXT = "Сайн байна уу, миний нэрийг ОронТТС гэдэг."  # @param {type:"string"}
LANG = "mn"  # @param ["mn", "kz"]
STEPS = 32  # @param {type:"slider", min:8, max:64, step:4}

OUTPUT_WAV = f"{OUTPUT_DIR}/output.wav"

if ckpts:
    !python scripts/infer.py \
        --checkpoint {LATEST_CKPT} \
        --text "{TEXT}" \
        --lang {LANG} \
        --steps {STEPS} \
        --output {OUTPUT_WAV}

In [ ]:
# Play audio in notebook
from IPython.display import Audio, display
import os

if os.path.exists(OUTPUT_WAV):
    display(Audio(OUTPUT_WAV))
else:
    print("No output file found. Run inference first.")

## 5. Voice Cloning (Optional)

Upload a 3–10 second reference WAV to clone a voice.

In [ ]:
# Upload reference audio
from google.colab import files

print("Upload a 3-10s WAV file as reference audio:")
uploaded = files.upload()

if uploaded:
    REF_AUDIO = list(uploaded.keys())[0]
    print(f"Reference audio: {REF_AUDIO}")

In [ ]:
# Synthesize with voice cloning
CLONE_TEXT = "Энэ бол дуу хоолой хуулбарлах туршилт юм."  # @param {type:"string"}

CLONE_OUTPUT = f"{OUTPUT_DIR}/clone_output.wav"

if ckpts and 'REF_AUDIO' in dir():
    !python scripts/infer.py \
        --checkpoint {LATEST_CKPT} \
        --text "{CLONE_TEXT}" \
        --lang mn \
        --ref-audio {REF_AUDIO} \
        --steps {STEPS} \
        --output {CLONE_OUTPUT}

    if os.path.exists(CLONE_OUTPUT):
        display(Audio(CLONE_OUTPUT))

## 6. Attribute-Based Synthesis (Optional)

Control voice properties without reference audio.

In [ ]:
ATTR_TEXT = "Өнөөдөр цаг агаар маш сайхан байна."  # @param {type:"string"}
ATTR_TOKENS = "[FEMALE],[YOUNG]"  # @param {type:"string"}

ATTR_OUTPUT = f"{OUTPUT_DIR}/attr_output.wav"

if ckpts:
    !python scripts/infer.py \
        --checkpoint {LATEST_CKPT} \
        --text "{ATTR_TEXT}" \
        --lang mn \
        --attr-tokens "{ATTR_TOKENS}" \
        --steps {STEPS} \
        --output {ATTR_OUTPUT}

    if os.path.exists(ATTR_OUTPUT):
        display(Audio(ATTR_OUTPUT))

## 7. Text Normalization Demo

Test number/date/currency normalization.

In [ ]:
from src.utils.number_norm import NumberNormalizer

mn = NumberNormalizer("mn")

examples = [
    "2026 оны 10-р сарын 12",
    "2026.10.12",
    "Үнэ: 5100₮",
    "14:30 цагт",
    "50% хөнгөлөлт",
    "XXI зуун",
    "-15°C хүйтэн",
    "+976 9911 2233",
]

for ex in examples:
    print(f"{ex:30s} → {mn.normalize_text(ex)}")